# Recon 6
Fuzz 9090 + 8012, read process info.

In [ ]:
import subprocess
def run(cmd, t=40):
    try:
        p = subprocess.run(cmd, shell=True, capture_output=True, timeout=t)
        return (p.stdout + p.stderr).decode("utf-8", "replace").replace("\x00", "\\x00")
    except Exception as e:
        return f"ERR: {e}"

print(run("cat /proc/7/cmdline 2>&1 | tr '\\0' ' '; echo; echo '--environ--'; tr '\\0' '\\n' < /proc/7/environ 2>&1 | head -30", 10))
print(run("ls -la /proc/7/ 2>&1 | head -30; ls -la /proc/7/cwd 2>&1; ls -la /proc/7/root 2>&1; ls -la /proc/7/fd 2>&1 | head -30", 10))
print(run("ls -la /proc/48/ 2>&1 | head -20; cat /proc/48/cmdline 2>&1 | tr '\\0' ' '; echo", 10))
print(run("ls -la /posit 2>&1; ls -la /posit/vivid-blender-live 2>&1; head -c 200 /posit/vivid-blender-live 2>&1 | xxd | head -3", 10))

In [ ]:
import subprocess
script = r'''
import urllib.request, socket
paths = ["/", "/health", "/healthz", "/metrics", "/debug", "/debug/pprof/", "/v1", "/v1/", "/api", "/api/", "/status", "/version", "/info", "/ping", "/ready", "/api/v1", "/api/v1/", "/fileops", "/file", "/files", "/s3", "/upload", "/download", "/bundle", "/content", "/contents", "/_internal", "/__internal", "/__debug__", "/grpc", "/grpc/", "/docs", "/openapi.json", "/favicon.ico"]
def hit(port, path):
    try:
        s = socket.create_connection(("127.0.0.1", port), timeout=2)
        s.send(("GET " + path + " HTTP/1.0\r\nHost: 127.0.0.1\r\n\r\n").encode())
        data = s.recv(400)
        s.close()
        line = data.split(b"\r\n",1)[0].decode("utf-8","replace")
        print(port, path, "->", line)
    except Exception as e:
        print(port, path, "FAIL:", e)
for path in paths:
    hit(9090, path)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=120)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])

In [ ]:
import subprocess
script = r'''
import urllib.request, socket
paths = ["/", "/__/health", "/__/status", "/__/metrics", "/__/queue", "/__/auth", "/__/login", "/__/api", "/__/v1", "/queue", "/queues", "/jobs", "/job", "/_/", "/__", "/__/", "/healthz", "/metrics", "/version", "/v1/", "/api/"]
def hit(port, path):
    try:
        s = socket.create_connection(("127.0.0.1", port), timeout=2)
        s.send(("GET " + path + " HTTP/1.0\r\nHost: 127.0.0.1\r\n\r\n").encode())
        data = s.recv(400)
        s.close()
        line = data.split(b"\r\n",1)[0].decode("utf-8","replace")
        print(port, path, "->", line)
    except Exception as e:
        print(port, path, "FAIL:", e)
for path in paths:
    hit(8012, path)
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=120)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])

In [ ]:
import subprocess
script = r'''
import socket
# full banner for 9090 root
s = socket.create_connection(("127.0.0.1", 9090), timeout=3)
s.send(b"GET / HTTP/1.1\r\nHost: 127.0.0.1\r\nConnection: close\r\n\r\n")
data = b""
while True:
    c = s.recv(4096)
    if not c: break
    data += c
print(repr(data[:3000]))
s.close()
# try POST and OPTIONS to 9090
for method in ["POST", "OPTIONS", "PUT"]:
    s = socket.create_connection(("127.0.0.1", 9090), timeout=3)
    s.send((method + " / HTTP/1.1\r\nHost: 127.0.0.1\r\nConnection: close\r\n\r\n").encode())
    d = s.recv(500)
    print(method, repr(d[:300]))
    s.close()
'''
p = subprocess.run(["python3", "-c", script], capture_output=True, timeout=60)
print(p.stdout.decode("utf-8", "replace"), p.stderr.decode("utf-8", "replace")[:3000])